In [ ]:
import cobra
from tqdm import tqdm
import warnings
from utils import functions as func

def test_create_stoichiometric_matrix_1():
    for name in ['textbook', 'ecoli', 'mini']:
        mod = cobra.test.create_test_model(name)

        S_mod = cobra.util.create_stoichiometric_matrix(mod)
        test_me = func.ME_Model('')
        test_me.add_metabolites([m.copy() for m in mod.metabolites])
        test_me.add_reactions([r.copy() for r in mod.reactions])
        S_me = test_me.create_stoichiometric_matrix(mu_val = float('nan'), array_type = 'numpy', inplace = False)
        if not np.array_equal(S_mod, S_me):
            raise ValueError('Stoichiometrc matrix not generated correctly')


def test_create_stoichiometric_matrix_2(me_model):
    '''
    Tests whether replacing mu with a value produces correct stoichiometric matrix
    '''
    mismatch = None
    S_me_1 = me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'numpy')

    reactions = [r.copy() for r in tqdm(me_model.reactions)]
    for r in reactions:
        if isinstance(r, func.ME_Reaction):
            if len(set(r.type).intersection(['translation', 'catalysis'])) > 0:
                r.replace_coefficient_mu(1)

    test_me = func.ME_Model('test')
    test_me.add_reactions(reactions)
    
    S_me_2 = test_me.create_stoichiometric_matrix(mu_val = float('nan'), inplace = False, array_type = 'numpy')
    
    if not np.array_equal(S_me_1, S_me_2):
        mismatch = np.argwhere(np.not_equal(S_me_1, S_me_2))
        warnings.warn('Mu val replacement fails in creating stoichiometric matrix')
    
    return S_me_1, S_me_2, mismatch

def test_create_stoichiometric_matrix_3(me_model):
    mismatch = None
    reactions_me = [r.copy() for r in tqdm(me_model.reactions)]
    reactions_mod = list()

    S_me = me_model.create_stoichiometric_matrix(mu_val = 1, inplace = False, array_type = 'numpy')

    for r in reactions_me:
        lb, ub = r.bounds
        if isinstance(r, func.ME_Reaction):
            if len(set(r.type).intersection(['translation', 'catalysis'])) > 0:
                r.replace_coefficient_mu(1)
            else:
                lb, ub = r.replace_bound_mu(1)

            new_r = cobra.Reaction(id = r.id, name = r.name, subsystem = r.subsystem, 
                               lower_bound = lb, upper_bound = ub)
            new_r.add_metabolites(r.metabolites.copy())
            new_r.gene_reaction_rule = r.gene_reaction_rule
        else:
            new_r = r.copy()
        reactions_mod.append(new_r)

    test_mod = cobra.Model('test')
    test_mod.add_reactions(reactions_mod)
    S_mod = cobra.util.create_stoichiometric_matrix(test_mod)

    if not np.array_equal(S_me, S_mod):
        mismatch = np.argwhere(np.not_equal(S_me, S_mod))
        warnings.warn('Stoichiometrc matrix not generated correctly')
    return S_me, S_mod, mismatch
    
def test_linear_solver(threshold = 1e-13):
    name_mapper = {'textbook': ['Biomass_Ecoli_core'], 'ecoli': ['BIOMASS_Ec_iJO1366_core_53p95M']}
    for name in ['textbook', 'ecoli', 'mini']:
        print(name)
        mod = cobra.test.create_test_model(name)
        mod_opt = mod.optimize().to_frame()

        test_me = func.ME_Model('')
        test_me.add_metabolites([m.copy() for m in mod.metabolites])
        test_me.add_reactions([r.copy() for r in mod.reactions])

        with func.HiddenPrints():
            sln, stat, _ = test_me.solve_lp(objective = {r_id:1 for r_id in name_mapper[name]}, 
                                            mu_val = float('nan'))
        if stat != 0:
            warnings.warn('The model fails to solve')
        precision = mod_opt[mod_opt.fluxes != abs(mod_opt.fluxes).min()].fluxes.abs().min()
        sln[abs(sln) < precision] = 0
        mod_opt['qminos_fluxes'] = sln[:len(mod.reactions)]

        diff = mod_opt[mod_opt.fluxes != mod_opt.qminos_fluxes]
        if diff[abs(diff.fluxes - diff.qminos_fluxes)>threshold].shape[0]>0:
            warnings.warn('There are different fluxes for GLPK and QMINOS solution')

        ir = test_me.infeasible_reactions(sln = sln, mu_val = float('nan'))
        if len(ir)>0:
            warnings.warn('There are unallowed fluxes for QMINOS solution')
        print('----------')    

 

In [ ]:
file_name = 
with open(file_name + '.pickle', 'rb') as handle:
    tme = pickle.load(handle)
    
# stoichiometric matrix
test_create_stoichiometric_matrix_1()
S_me_1, S_me_2, mismatch = test_create_stoichiometric_matrix_2(me_model)
S_me, S_mod, mismatch_ = test_create_stoichiometric_matrix_3(me_model)
# solver
test_linear_solver(threshold = 1e-13)